<a href="https://colab.research.google.com/github/K-Brant/-distilbert-cybersecurity/blob/main/CyberSecurityAwareness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn

In [4]:
# Install required libraries (run this cell first)
!pip install transformers datasets evaluate accelerate scikit-learn pandas

import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
import os

# Create results directory
os.makedirs("./results", exist_ok=True)

# Load your Ghana GOLD data
df = pd.read_csv("Ghana_GOLD_Labelled_144.csv")

# Check the data
print(f"Total rows: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# Define columns
text_column = "text"
label_column = "label"

# Convert to Dataset
dataset = Dataset.from_pandas(df)

# Split into train and test (80% train, 20% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

# Label mapping (0=Low, 1=Medium, 2=High)
id2label = {0: "Low", 1: "Medium", 2: "High"}
label2id = {"Low": 0, "Medium": 1, "High": 2}

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples[text_column], padding="max_length", truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Load model with 3 classes
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,                     # <-- CHANGED: 3 classes (Low/Medium/High)
    id2label=id2label,
    label2id=label2id
)

# Define metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1", average="weighted")
precision = evaluate.load("precision", average="weighted")
recall = evaluate.load("recall", average="weighted")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"],
        "precision": precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall.compute(predictions=predictions, references=labels)["recall"]
    }

# Training arguments - ENGINEERED MODEL with focal loss equivalent
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,              # More epochs for small dataset
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    optim="adamw_torch"
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("\n" + "="*50)
print("ENGINEERED MODEL TRAINING - FOCAL-DISTILBERT ON GHANA GOLD")
print("="*50)
trainer.train()

print("\n" + "="*50)
print("EVALUATING MODEL...")
print("="*50)
results = trainer.evaluate()

print("\n" + "="*50)
print("ENGINEERED MODEL RESULTS - GHANA GOLD FIELD DATA (n=219)")
print("="*50)
print(f"Accuracy:  {results['eval_accuracy']:.4f}")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"F1-Score:  {results['eval_f1']:.4f}")
print("="*50)

Total rows: 100
Label distribution:
label
0    34
1    33
2    33
Name: count, dtype: int64
Train size: 80
Test size: 20


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



ENGINEERED MODEL TRAINING - FOCAL-DISTILBERT ON GHANA GOLD


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].